In [44]:
from pgmpy.models import MarkovNetwork
from pgmpy.models import FactorGraph
from pgmpy.factors.discrete import DiscreteFactor
from pgmpy.inference import BeliefPropagation
import numpy as np

In [65]:
# Define the Markov Network (example with 3 indicators)
model = MarkovNetwork([('X1', 'X2'), ('X2', 'X3')])

# Clique potentials with contingency tables (conditional on Y=1)
psi_X1 = DiscreteFactor(['X1'], [2], [0.8,0.2])  # P(X1 | Y=1)
psi_X2 = DiscreteFactor(['X2'], [2], [0.6,0.4])  # P(X2 | Y=1)
psi_X3 = DiscreteFactor(['X3'], [2], [0.7,0.3])  # P(X3 | Y=1)
psi_X1_X2 = DiscreteFactor(['X1', 'X2'], [2, 2], [0.6, 0.1, 0.2, 0.1])  # P(X1, X2 | Y=1)
psi_X2_X3 = DiscreteFactor(['X2', 'X3'], [2, 2], [0.5, 0.2, 0.1, 0.2])  # P(X2, X3 | Y=1)
model.add_factors(psi_X1, psi_X2, psi_X3, psi_X1_X2, psi_X2_X3)

In [66]:
print(model)

MarkovNetwork with 3 nodes and 2 edges


In [67]:
inference = BeliefPropagation(model)

In [68]:
# Calculate censored indicators headcount ratios
h_X1 = inference.query(['X1']).values[1]  # P(X1=1 | Y=1)
h_X2 = inference.query(['X2']).values[1]  # P(X2=1 | Y=1)
h_X3 = inference.query(['X3']).values[1]  # P(X3=1 | Y=1)
print(f"Baseline P(X1=1): {h_X1:.3f},  P(X2=1): {h_X2:.3f}, P(X3=1): {h_X3:.3f}")

Baseline P(X1=1): 0.082,  P(X2=1): 0.039, P(X3=1): 0.159


In [69]:
M0 = 1/3 * h_X1 + 1/3 * h_X2 + 1/3 * h_X2
print(f"Baseline M0: {M0:.3f}")

Baseline M0: 0.053


In [70]:
psi_X1_X2_new = DiscreteFactor(['X1', 'X2'], [2, 2], [0.6, 0.25, 0.1, 0.05])  # Updated potential
model.remove_factors(psi_X1_X2)
model.add_factors(psi_X1_X2_new)

In [71]:
inference = BeliefPropagation(model)

In [72]:
# Updated marginals
h_X1_new = inference.query(['X1']).values[1]  # P(X1=1 | Y=1) after intervention
h_X2_new = inference.query(['X2']).values[1]  # P(X2=1 | Y=1) after intervention
h_X3_new = inference.query(['X3']).values[1]  # P(X3=1 | Y=1) after intervention
print(f"Post-intervention P(X1=1): {h_X1_new:.3f}, P(X2=1): {h_X2_new:.3f}, P(X3=1): {h_X3_new:.3f}")

Post-intervention P(X1=1): 0.041, P(X2=1): 0.082, P(X3=1): 0.172


In [73]:
M0_new = 1/3 * h_X1_new + 1/3 * h_X2_new + 1/3 * h_X3_new
print(f"Post-intervention M0: {M0_new:.3f}")

Post-intervention M0: 0.098
